# URL 수집 결과 통계 (Colab용)

Google Drive에 저장된 `링크_*.json` 파일을 읽어서 query별/월별 URL 개수를 확인합니다.


In [6]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import json
import unicodedata
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path('/content/drive/MyDrive/Text-data-Analysis_26-Spring')
CANDIDATE_DIRS = [
    PROJECT_DIR / 'notebook' / 'crawling' / 'data',
]

for data_dir in CANDIDATE_DIRS:
    print('-', data_dir, '존재' if data_dir.exists() else '없음')


- /content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data 존재


In [8]:
def normalized_name(path):
    # Drive/FUSE 환경에서 한글 파일명이 NFC/NFD 형태로 다를 수 있어 정규화
    return unicodedata.normalize('NFC', path.name)


def is_link_json(path):
    name = normalized_name(path)
    return name.startswith('링크_') and name.endswith('.json')


def parse_link_file_name(path):
    # 파일명 형식: 링크_{query}_{YYMMDD}_{YYMMDD}.json
    stem = unicodedata.normalize('NFC', path.stem)
    if not stem.startswith('링크_'):
        return None

    rest = stem.removeprefix('링크_')
    query, start, end = rest.rsplit('_', 2)
    start_ym = f'20{start[:2]}.{start[2:4]}'
    end_ym = f'20{end[:2]}.{end[2:4]}'

    return query, start, end, start_ym, end_ym


json_files = []
for data_dir in CANDIDATE_DIRS:
    if data_dir.exists():
        json_files.extend(data_dir.glob('*.json'))

# 후보 폴더에서 json도 못 찾으면 프로젝트 폴더 전체를 한 번 더 탐색
if not json_files and PROJECT_DIR.exists():
    print('후보 폴더에서 JSON 파일을 못 찾아 프로젝트 폴더 전체를 탐색합니다.')
    json_files = list(PROJECT_DIR.rglob('*.json'))

json_files = sorted(set(json_files))
print(f'발견한 JSON 파일: {len(json_files)}개')
for path in json_files[:30]:
    print('-', repr(normalized_name(path)), 'in', path.parent)
if len(json_files) > 30:
    print(f'... 외 {len(json_files) - 30}개')

link_files = [path for path in json_files if is_link_json(path)]
print(f'발견한 링크 JSON 파일: {len(link_files)}개')
for path in link_files[:20]:
    print('-', repr(normalized_name(path)), 'in', path.parent)
if len(link_files) > 20:
    print(f'... 외 {len(link_files) - 20}개')

rows = []
for path in link_files:
    parsed = parse_link_file_name(path)
    if parsed is None:
        continue

    query, start, end, start_ym, end_ym = parsed

    with path.open('r', encoding='utf-8') as f:
        urls = json.load(f)

    rows.append({
        'query': query,
        'period': f'{start}_{end}',
        'start_ym': start_ym,
        'end_ym': end_ym,
        'url_count': len(urls),
        'unique_url_count': len(set(urls)),
        'duplicate_count': len(urls) - len(set(urls)),
        'file_size_kb': round(path.stat().st_size / 1024, 1),
        'file_name': normalized_name(path),
        'file_path': str(path),
    })

stats_df = pd.DataFrame(rows)

if stats_df.empty:
    raise ValueError('링크 JSON 파일을 찾지 못했습니다. 위에 출력된 JSON 파일명과 실제 Drive 저장 위치를 확인하세요.')

stats_df = stats_df.sort_values(['query', 'period']).reset_index(drop=True)
stats_df


발견한 JSON 파일: 51개
- '링크_KT_250901_250930.json' in /content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data
- '링크_KT_251001_251031.json' in /content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data
- '링크_KT_251101_251130.json' in /content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data
- '링크_KT_251201_251231.json' in /content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data
- '링크_KT_260101_260131.json' in /content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data
- '링크_LG U+_250801_250831.json' in /content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data
- '링크_LG U+_250901_250930.json' in /content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data
- '링크_LG U+_251001_251031.json' in /content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data
- '링크_LG U+_251101_251130.json' in /content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data
- '링크_LG

,query,period,start_ym,end_ym,url_count,unique_url_count,duplicate_count,file_size_kb,file_name,file_path
0,KT,250901_250930,2025.09,2025.09,7900,7900,0,516.9,링크_KT_250901_250930.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
1,KT,251001_251031,2025.10,2025.10,3688,3688,0,241.3,링크_KT_251001_251031.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
2,KT,251101_251130,2025.11,2025.11,4442,4442,0,290.6,링크_KT_251101_251130.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
3,KT,251201_251231,2025.12,2025.12,5018,5018,0,328.3,링크_KT_251201_251231.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
4,KT,260101_260131,2026.01,2026.01,2859,2859,0,187.1,링크_KT_260101_260131.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
5,LG U+,250801_250831,2025.08,2025.08,478,478,0,31.3,링크_LG U+_250801_250831.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
6,LG U+,250901_250930,2025.09,2025.09,557,557,0,36.4,링크_LG U+_250901_250930.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
7,LG U+,251001_251031,2025.10,2025.10,534,534,0,34.9,링크_LG U+_251001_251031.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
8,LG U+,251101_251130,2025.11,2025.11,567,567,0,37.1,링크_LG U+_251101_251130.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
9,LG U+,251201_251231,2025.12,2025.12,645,645,0,42.2,링크_LG U+_251201_251231.json,/content/drive/MyDrive/Text-data-Analysis_26-S...


In [9]:
# query별/월별 URL 개수 표
pivot_df = stats_df.pivot_table(
    index='query',
    columns='start_ym',
    values='unique_url_count',
    aggfunc='sum',
    fill_value=0,
)

pivot_df['total'] = pivot_df.sum(axis=1)
pivot_df


start_ym,2025.04,2025.05,2025.06,2025.07,2025.08,2025.09,2025.10,2025.11,2025.12,2026.01,total
query,,,,,,,,,,,
KT,0,0,0,0,0,7900,3688,4442,5018,2859,23907
LG U+,0,0,0,0,478,557,534,567,645,0,2781
LG유플러스,0,0,0,0,1400,1787,1428,1482,1435,0,7532
SKT,4454,5696,1473,2101,1788,0,0,0,0,0,15512
SK텔레콤,5361,6474,2384,2993,2810,0,0,0,0,0,20022


In [10]:
# query별 요약 통계
summary_df = (
    stats_df
    .groupby('query', as_index=False)
    .agg(
        months=('period', 'count'),
        total_urls=('unique_url_count', 'sum'),
        min_month_urls=('unique_url_count', 'min'),
        max_month_urls=('unique_url_count', 'max'),
        avg_month_urls=('unique_url_count', 'mean'),
        duplicate_urls=('duplicate_count', 'sum'),
    )
)
summary_df['avg_month_urls'] = summary_df['avg_month_urls'].round(1)
summary_df


,query,months,total_urls,min_month_urls,max_month_urls,avg_month_urls,duplicate_urls
0,KT,5,23907,2859,7900,4781.4,0
1,LG U+,5,2781,478,645,556.2,0
2,LG유플러스,5,7532,1400,1787,1506.4,0
3,SKT,5,15512,1473,5696,3102.4,0
4,SK텔레콤,5,20022,2384,6474,4004.4,0


In [11]:
# 월별 URL 수가 적은 파일부터 확인
stats_df.sort_values('unique_url_count').reset_index(drop=True)


,query,period,start_ym,end_ym,url_count,unique_url_count,duplicate_count,file_size_kb,file_name,file_path
0,LG U+,250801_250831,2025.08,2025.08,478,478,0,31.3,링크_LG U+_250801_250831.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
1,LG U+,251001_251031,2025.10,2025.10,534,534,0,34.9,링크_LG U+_251001_251031.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
2,LG U+,250901_250930,2025.09,2025.09,557,557,0,36.4,링크_LG U+_250901_250930.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
3,LG U+,251101_251130,2025.11,2025.11,567,567,0,37.1,링크_LG U+_251101_251130.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
4,LG U+,251201_251231,2025.12,2025.12,645,645,0,42.2,링크_LG U+_251201_251231.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
5,LG유플러스,250801_250831,2025.08,2025.08,1400,1400,0,91.6,링크_LG유플러스_250801_250831.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
6,LG유플러스,251001_251031,2025.10,2025.10,1428,1428,0,93.4,링크_LG유플러스_251001_251031.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
7,LG유플러스,251201_251231,2025.12,2025.12,1435,1435,0,93.9,링크_LG유플러스_251201_251231.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
8,SKT,250601_250630,2025.06,2025.06,1473,1473,0,96.4,링크_SKT_250601_250630.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
9,LG유플러스,251101_251130,2025.11,2025.11,1482,1482,0,97.0,링크_LG유플러스_251101_251130.json,/content/drive/MyDrive/Text-data-Analysis_26-S...
